In [11]:
from dotenv import load_dotenv
from IPython.display import Markdown, display
import os

In [2]:
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

from openai import OpenAI
openai = OpenAI(api_key=api_key)


In [3]:
system_prompt = """
You are an experienced music instructor. You are given the text content of a website that a musician has searched, and sometimes a question about it.

Your job:
- Decide whether the website content is related to music, such as instruments, music theory, ear training, performance, composition, music production or music certification exams.
- If it is related, help the musician: answer their question, or explain how the content applies to their music study or certification preparation. Base your answer on the website content. If the content does not contain the answer, say so instead of guessing; you may add general music knowledge, clearly labelled as coming from outside the website.
- If it is not related to music, reply with a short, snarky remark saying the site has nothing to do with music. Do not answer the question in that case.

Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [4]:
user_prompt_prefix = """
Here are the contents of a website.
Analyze from a musical teacher standpoint and give your viewpoint.

"""

In [5]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [6]:
import requests
from bs4 import BeautifulSoup

def fetch_website_contents(url):

    # Download the page
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()

    # Parse the HTML
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract the page title
    title = soup.title.string if soup.title and soup.title.string else "No title found"
    result = f"Title: {title}\n\nLinks:\n"

    # Extract all links
    for link in soup.find_all("a", href=True):
        text = link.get_text(strip=True)
        result += f"{text} -> {link['href']}\n"
    return result


In [7]:
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [8]:

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://en.wikipedia.org/wiki/Guitar")